# Explore the data

In [1]:
import pandas as pd

train = pd.read_csv("data/train.csv")
print(train.shape)
train.head()

(1561, 3)


,oare_id,transliteration,translation
0,004a7dbd-57ce-46f8-9691-409be61c676e,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-{d}IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ..."
1,0064939c-59b9-4448-a63d-34612af0a1b5,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,Itūr-ilī has received one textile of ordinary ...
2,0073f2c0-524c-4bbf-915a-8c1772a4fb98,TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR,<gap> he did not give you a textile. He return...
3,009fb838-8038-42bc-ad34-5f795b3840ee,KIŠIB šu-{d}EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...,"Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū..."
4,00aa1c55-c80c-4346-a159-73ad43ab0ff7,um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...,From Šukkutum to Ištar-lamassī and Nitahšušar:...


In [ ]:
# Check number of rows
print(f"Training Examples: {len(train)}") 

Training Examples: 1561


In [4]:
# Check average length of texts

train['transliteration_len'] = train['transliteration'].str.split().str.len()
train['translation_len'] = train['translation'].str.split().str.len()

print(f"Avg Akkadian length: {train['transliteration_len'].mean():.1f} words")
print(f"Avg English length: {train['translation_len'].mean():.1f} words")
print(f"Max Akkadian length: {train['transliteration_len'].max()} words")
print(f"Max English length: {train['translation_len'].max()} words")

Avg Akkadian length: 56.2 words
Avg English length: 90.0 words
Max Akkadian length: 150 words
Max English length: 739 words


# OBSERVATION

### Dataset Structure

train.csv — 3 columns: oare_id, transliteration, translation
test.csv — 5 columns: id, text_id, line_start, line_end, transliteration
Train is document-level (whole tablets), test is sentence-level (fragments)
text_id groups test sentences that come from the same tablet
line_start / line_end tell you which lines of that tablet each sentence covers

### What The Texts Are About

3,800-year-old merchant records from ancient Turkey (Kanesh/Kültepe)
Legal documents — contracts, business letters, debt records
Themes: silver, copper, textiles, donkeys, debts, interest rates, legal disputes

### Two Languages Mixed Together
The Akkadian transliteration is actually two languages mixed together:

Akkadian syllables — written in lowercase hyphenated form e.g. ma-nu-ba-lúm-a-šur
Sumerian logograms — written in ALL CAPS, borrowed like abbreviations

KIŠIB = seal (signature)
DUMU = son of
KÙ.BABBAR = silver
TÚG = textile
ANŠE = donkey



This matters for the model — the tokenizer needs to handle both writing systems.

### Akkadian Text Conventions

{d} = deity determinative (silent classifier — "next word is a god's name")
{ki} = place name determinative
{f} = female person determinative
[ ] = clearly broken signs
˹ ˺ = partially broken signs
<gap> = missing text

### Formatting Inconsistencies Between Train and Test

Train uses Ḫ/ḫ, test uses H/h → needs normalization
Train uses {ki}, test uses (ki) → needs normalization

### Text Length Statistics
AkkadianEnglishAverage length56.2 words90.0 wordsMax length150 words739 words

- English is longer than Akkadian — logograms compress meaning
- Max English of 739 words is a red flag for models with 512 token limits
- The Akkadian transliteration is actually two languages mixed together:
    - Akkadian syllables — written in lowercase hyphenated form e.g. ma-nu-ba-lúm-a-šur
    - Sumerian logograms — written in ALL CAPS, borrowed like abbreviation 


In [7]:
# Average test sentence length
test = pd.read_csv("data/test.csv")
print(f"Test examples: {len(test)}")

test['trans_len'] = test['transliteration'].str.split().str.len()
print(f"\nAvg test Akkadian length: {test['trans_len'].mean():.1f} words")
print(f"Max test Akkadian length: {test['trans_len'].max()} words")

Test examples: 4

Avg test Akkadian length: 21.2 words
Max test Akkadian length: 34 words


In [8]:
# Count total number of rows(each row = one tablet/document)
print(f"Total training documents: {len(train)}")

# Check if there are any missing values in any column
# isnull() marks each cell True if empty, sum() counts the Trues
print(f"\Missing values:\n {train.isnull().sum()}")

Total training documents: 1561
\Missing values:
 oare_id                0
transliteration        0
translation            0
transliteration_len    0
translation_len        0
dtype: int64


In [ ]:
""" OBSERVE ONE FULL TEXT WITHOUT TRUNCATION"""

# Pick one row and print the full transliteration and translation
# iloc[0] selects the first row by position
example = train.iloc[0]

print("=== AKKADIAN ===")
# Print the full transliteration text
print(example['transliteration'])

print("=== ENGLISH ===")
# Print the full English translation
